# Presentation clips — annotated movie + PDF stills

Make the **presentation deliverables** for one session:

1. **Generate the annotated eye/face clip** — the labelled movie (ROI boxes, BLINK / EYE SQUINT /
   LICKING / GROOMING / WHISKING pills, pupil-fit inset, radius sparkline), the single-session analog of
   `JPAS_0231/test/trial15_saccade_6770_8890_annotated.mp4`. Driven by
   `common/detectors/annotate_clip.py` (style auto-detected: `approx` iris vs `stable` iris).
2. **Get an already-generated clip** — list the clips already in `<session>/test/` and play one.
3. **Grab a PDF (or PNG) still from a clip** — pull one frame out of a clip (e.g. session frame 6920)
   and save it as a vector-container PDF for a slide.

> **Dependency.** The annotated clip needs the **detector outputs** (`opticflow/pupil_track.npz`,
> `eye_events.npz`, and — if present — `mouth_state.npz` / `whisker.npz`). Run the full per-session
> pipeline first (flow → pupil → eye events → …), not just the flow/pupil steps of notebook 0.

In [ ]:
import sys, json, re
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from IPython.display import Video, display

# locate common/ (this notebook lives in optic_flow/presentations/) by searching upward
_HERE = Path.cwd()
_cands = [_HERE, _HERE / 'common', _HERE.parent / 'common', _HERE.parent.parent / 'common',
          _HERE.parent.parent.parent / 'common']
_COMMON = next((c for c in _cands if (c / 'compute_roi_flow.py').exists()), None)
assert _COMMON is not None, f'cannot locate common/ from {_HERE}'
sys.path.insert(0, str(_COMMON)); sys.path.insert(0, str(_COMMON / 'detectors'))
import annotate_clip                     # the eye/face annotated clip
try:
    import annotate_mouth_clip           # optional: the lick-vs-groom mouth clip
except Exception:
    annotate_mouth_clip = None
print('common ->', _COMMON)

In [ ]:
# ── CONFIG: point at ONE session folder (same layout as notebook 0) ───────────
MAIN_DIR = '/path/to/MAIN_DIR'    # <-- server root that holds the animal folders
MOUSE_ID = 'MOUSE_ID'             # <-- animal folder name
date     = ''                     # <-- session sub-folder (recording date/time); '' = none

SESSION_DIR = (Path(MAIN_DIR) / MOUSE_ID / date).resolve()
assert SESSION_DIR.exists(), f'session folder does not exist: {SESSION_DIR}  (set MAIN_DIR / MOUSE_ID / date)'
assert (SESSION_DIR / 'session.json').exists(), \
    f'no session.json in {SESSION_DIR} -- run notebook 0 (build the session contract) first'
_S = json.load(open(SESSION_DIR / 'session.json'))
TEST_DIR = SESSION_DIR / 'test'                       # annotate_clip writes here
PRES_DIR = SESSION_DIR / 'presentations'              # PDF stills go here by default
print('session   :', SESSION_DIR)
print('mouse     :', _S.get('mouse_id', MOUSE_ID), '| task:', _S.get('task_type'),
      '| fps:', round(_S.get('fps', 0), 3))
print('clips dir :', TEST_DIR)
print('stills dir:', PRES_DIR)

## 1 — Clips already generated

What annotated clips are already sitting in `<session>/test/`? (Pick one of these to pull a still from
in step 3, or generate a fresh one in step 2.)

In [ ]:
def list_clips():
    clips = sorted(TEST_DIR.glob('*.mp4')) if TEST_DIR.exists() else []
    if not clips:
        print('no clips in', TEST_DIR, '-- generate one in step 2'); return []
    for p in clips:
        cap = cv2.VideoCapture(str(p)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
        m = re.findall(r'_(\d+)_(\d+)', p.stem)
        span = f'  session frames {m[-1][0]}-{m[-1][1]}' if m else ''
        print(f'  {p.name}   ({n} frames){span}')
    return clips

_clips = list_clips()

## 2 — Generate the annotated eye/face clip

Pick a **session-frame window** `[LO, HI]` (or a trial, below) and a **style** (`None` = auto: `approx`
for an approximate iris like JPAS_0168, `stable` when dil/con spans exist). Writes
`<session>/test/eye_clip_<style>_<LO>_<HI>.mp4` and returns its path.

In [ ]:
LO, HI = 5000, 5200      # session-frame window for the clip
STYLE  = None            # None = auto | 'approx' | 'stable'

def trial_window(n, pad=0):
    '''(start_frame, end_frame) for trial n from df_trials_clean.pkl, if present -- so you can clip a
    whole trial instead of guessing frames.  clip = annotate_eye(*trial_window(15))'''
    import pandas as pd
    dfp = SESSION_DIR / 'df_trials_clean.pkl'
    assert dfp.exists(), 'no df_trials_clean.pkl -- give LO/HI by frame instead'
    row = pd.read_pickle(dfp).loc[n]
    s = int(row.get('start_frame', row.get('clean_start'))); e = int(row.get('end_frame', row.get('clean_end')))
    return max(0, s - pad), e + pad

def annotate_eye(lo, hi, style=STYLE):
    '''Render the annotated eye/face clip for [lo, hi]; returns the .mp4 path.'''
    try:
        return annotate_clip.main(str(SESSION_DIR), int(lo), int(hi), style)
    except FileNotFoundError as ex:
        raise FileNotFoundError(f'{ex}\n-> the annotated clip needs the detector outputs '
                                '(pupil_track.npz, eye_events.npz, ...). Run the full pipeline first.')

CLIP = annotate_eye(LO, HI)
print('clip ->', CLIP)

*(Optional — the **mouth** lick-vs-groom clip, if `annotate_mouth_clip` and its inputs exist:
`annotate_mouth_clip.main(str(SESSION_DIR), LO, HI)`.)*

## 3 — Play a clip in the notebook

Embed a short clip inline. Big clips are just reported by path (base64-embedding a large mp4 is heavy);
open those from `<session>/test/` in a video player.

In [ ]:
def play(clip, max_mb=25):
    clip = Path(clip)
    mb = clip.stat().st_size / 1e6
    print(f'{clip.name}  ({mb:.1f} MB)')
    if mb <= max_mb:
        display(Video(str(clip), embed=True))
    else:
        print('  too big to embed inline -- open it from', clip.parent)

play(CLIP)      # or play(_clips[0]) to view an already-generated one

## 4 — Grab a PDF (or PNG) still from a clip

Pull ONE frame out of a clip and save it. `frame` is a **session frame index**; the clip's start frame
is parsed from its `_<lo>_<hi>_` filename (so `frame=6920` on a 6770–8890 clip maps to the right
picture). Pass `start=` to override, or `start=0` to treat `frame` as clip-local. Default output is a
**PDF** in `<session>/presentations/`; use `fmt='png'` / `save_path=` / `outdir=` to redirect.

The clip's frame is already annotated, so the still needs no extra drawing — it just embeds that frame.

In [ ]:
def pdf_from_clip(clip, frame, start=None, outdir=None, save_path=None, fmt='pdf', show=True):
    '''Save session-`frame` of `clip` as a PDF/PNG still. start = the clip's first SESSION frame
    (parsed from the filename if None; pass 0 to treat `frame` as clip-local).'''
    clip = Path(clip)
    if start is None:
        m = re.findall(r'_(\d+)_(\d+)', clip.stem); start = int(m[-1][0]) if m else 0
    local = int(frame) - int(start)
    cap = cv2.VideoCapture(str(clip)); nfr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if not (0 <= local < nfr):
        cap.release()
        raise IndexError(f'session frame {frame} -> clip-local {local} is outside [0, {nfr}) '
                         f'(clip start={start}). Pass start= or pick a frame inside the clip.')
    cap.set(cv2.CAP_PROP_POS_FRAMES, local); ok, im = cap.read(); cap.release()
    if not ok or im is None:
        raise IOError(f'cannot read local frame {local} of {clip.name}')
    ext = (fmt or 'png').lstrip('.')
    if save_path:
        p = Path(save_path)
        if not p.suffix: p = p / f'{clip.stem}_f{frame}.{ext}'
    else:
        p = (Path(outdir) if outdir else PRES_DIR) / f'{clip.stem}_f{frame}.{ext}'
    p.parent.mkdir(parents=True, exist_ok=True)
    rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', ''):
        cv2.imwrite(str(p), im)
    else:                                   # .pdf / .svg -> embed the (already-annotated) raster frame
        fig = plt.figure(figsize=(w / 100, h / 100), dpi=100); ax = fig.add_axes([0, 0, 1, 1])
        ax.imshow(rgb); ax.axis('off'); fig.savefig(str(p), bbox_inches='tight', pad_inches=0); plt.close(fig)
    print('saved', p)
    if show:
        plt.figure(figsize=(12, 6.8)); plt.imshow(rgb); plt.axis('off')
        plt.title(f"{clip.stem}   session frame {frame}"); plt.show()
    return p

# e.g. session frame 6920 from a trial15_saccade_6770_8890 clip -> a PDF slide:
pdf_from_clip(CLIP, 5030)                 # <- edit the frame; add fmt='png' / save_path='...' to redirect